# 🌊 NumCompute-Stream — Streaming ML Demo

**Assignment 2.2 | stream_demo.ipynb**

This notebook demonstrates the full NumCompute-Stream pipeline:

1. Load the Iris dataset from CSV using custom `io.py`
2. Split into chunks to simulate a **streaming data setting**
3. Train models incrementally using `.partial_fit()` on each chunk
4. Log and visualise key metrics over time using `visualise.py`
5. Compare single Decision Tree vs Random Forest under streaming
6. Run benchmark comparison

**Only NumPy and matplotlib used — no scikit-learn, no pandas.**

## 0. Setup & Imports

In [1]:
import sys
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for notebook compatibility
import matplotlib.pyplot as plt

# Make sure numcompute_stream is importable
sys.path.insert(0, os.path.abspath('..'))

# NumCompute-Stream modules
from numcompute_stream.trees import DecisionTreeClassifier
from numcompute_stream.ensemble import RandomForestClassifier, BaggingClassifier
from numcompute_stream.streaming import StreamTrainer, chunk_data
from numcompute_stream.metrics import (
    StreamingAccuracy, StreamingPrecisionRecallF1,
    StreamingConfusionMatrix, RollingAccuracy
)
from numcompute_stream.pipeline import StreamingPipeline
from numcompute_stream import visualise

print('✅ All imports successful!')
print(f'NumPy version : {np.__version__}')
print(f'Python version: {sys.version.split()[0]}')

✅ All imports successful!
NumPy version : 1.26.2
Python version: 3.11.4


## 1. Load Dataset from CSV using Custom io.py

We use our own custom CSV loader — **no pandas allowed!**

In [2]:
def load_csv(filepath):
    """
    Custom CSV loader using only plain Python + NumPy.
    Returns X (features) and y (labels) as NumPy arrays.
    """
    with open(filepath, 'r') as f:
        lines = f.read().strip().split('\n')

    header = lines[0].split(',')
    print(f'Columns: {header}')

    rows = []
    for line in lines[1:]:
        values = line.split(',')
        rows.append([float(v) for v in values])

    data = np.array(rows)
    X = data[:, :-1]   # All columns except last = features
    y = data[:, -1].astype(int)  # Last column = label

    return X, y, header[:-1]


# Load the Iris dataset
X, y, feature_names = load_csv('iris.csv')

print(f'\n📊 Dataset loaded successfully!')
print(f'   Shape     : {X.shape}')
print(f'   Features  : {feature_names}')
print(f'   Classes   : {np.unique(y)} → 0=Setosa, 1=Versicolor, 2=Virginica')
print(f'   Class dist: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'\nFirst 5 rows:')
for i in range(5):
    print(f'  X={X[i]}  y={y[i]}')

Columns: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

📊 Dataset loaded successfully!
   Shape     : (150, 4)
   Features  : ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
   Classes   : [0 1 2] → 0=Setosa, 1=Versicolor, 2=Virginica
   Class dist: {0: 50, 1: 50, 2: 50}

First 5 rows:
  X=[5.1 3.5 1.4 0.2]  y=0
  X=[4.9 3.  1.4 0.2]  y=0
  X=[4.7 3.2 1.3 0.2]  y=0
  X=[4.6 3.1 1.5 0.2]  y=0
  X=[5.  3.6 1.4 0.2]  y=0


## 2. Simulate Streaming — Split Into Chunks

Real streaming means data arrives in small pieces over time.
We simulate this by splitting our dataset into chunks of 15 samples each.

In [3]:
CHUNK_SIZE = 15

# Shuffle data first (like real streaming — data arrives randomly)
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
X_shuffled = X[idx]
y_shuffled = y[idx]

# Split into chunks
chunks = list(chunk_data(X_shuffled, y_shuffled,
                         chunk_size=CHUNK_SIZE,
                         shuffle=False))

print(f'🌊 Streaming simulation setup:')
print(f'   Total samples : {len(X)}')
print(f'   Chunk size    : {CHUNK_SIZE}')
print(f'   Total chunks  : {len(chunks)}')
print()
for i, (Xc, yc) in enumerate(chunks):
    print(f'   Chunk {i+1:2d}: {Xc.shape[0]} samples | '
          f'classes present: {np.unique(yc).tolist()}')

🌊 Streaming simulation setup:
   Total samples : 150
   Chunk size    : 15
   Total chunks  : 10

   Chunk  1: 15 samples | classes present: [0, 1, 2]
   Chunk  2: 15 samples | classes present: [0, 1, 2]
   Chunk  3: 15 samples | classes present: [0, 1, 2]
   Chunk  4: 15 samples | classes present: [0, 1, 2]
   Chunk  5: 15 samples | classes present: [0, 1, 2]
   Chunk  6: 15 samples | classes present: [0, 1, 2]
   Chunk  7: 15 samples | classes present: [0, 1, 2]
   Chunk  8: 15 samples | classes present: [0, 1, 2]
   Chunk  9: 15 samples | classes present: [0, 1, 2]
   Chunk 10: 15 samples | classes present: [0, 1, 2]


## 3. Train Single Decision Tree — Streaming with partial_fit()

In [4]:
print('=' * 55)
print('  Training: Decision Tree (streaming)')
print('=' * 55)

tree_model = DecisionTreeClassifier(
    max_depth=5,
    criterion='gini',
    min_samples_split=2,
    random_state=42
)

tree_trainer = StreamTrainer(tree_model, verbose=True)

# Train chunk by chunk — simulating streaming
for X_chunk, y_chunk in chunks:
    tree_trainer.fit_chunk(X_chunk, y_chunk)

tree_summary = tree_trainer.summary()
print(f'\n📈 Decision Tree Summary:')
for k, v in tree_summary.items():
    print(f'   {k:<30}: {v}')

  Training: Decision Tree (streaming)
[Chunk   0] n=   15 | metric=0.0000 | cumulative=0.0000 | time=0.0109s | mem=0.6KB
[Chunk   1] n=   15 | metric=0.8667 | cumulative=0.4333 | time=0.0120s | mem=1.2KB
[Chunk   2] n=   15 | metric=0.8667 | cumulative=0.5778 | time=0.0085s | mem=1.8KB
[Chunk   3] n=   15 | metric=1.0000 | cumulative=0.6833 | time=0.0081s | mem=2.3KB
[Chunk   4] n=   15 | metric=1.0000 | cumulative=0.7467 | time=0.0097s | mem=2.9KB
[Chunk   5] n=   15 | metric=1.0000 | cumulative=0.7889 | time=0.0088s | mem=3.5KB
[Chunk   6] n=   15 | metric=0.9333 | cumulative=0.8095 | time=0.0119s | mem=4.1KB
[Chunk   7] n=   15 | metric=1.0000 | cumulative=0.8333 | time=0.0125s | mem=4.7KB
[Chunk   8] n=   15 | metric=0.9333 | cumulative=0.8444 | time=0.0144s | mem=5.3KB
[Chunk   9] n=   15 | metric=1.0000 | cumulative=0.8600 | time=0.0148s | mem=5.9KB

📈 Decision Tree Summary:
   n_chunks                      : 10
   total_samples                 : 150
   mean_metric               

## 4. Train Random Forest — Streaming with partial_fit()

In [5]:
print('=' * 55)
print('  Training: Random Forest (streaming)')
print('=' * 55)

rf_model = RandomForestClassifier(
    n_estimators=10,
    max_depth=5,
    criterion='gini',
    max_features='sqrt',
    random_state=42
)

rf_trainer = StreamTrainer(rf_model, verbose=True)

for X_chunk, y_chunk in chunks:
    rf_trainer.fit_chunk(X_chunk, y_chunk)

rf_summary = rf_trainer.summary()
print(f'\n📈 Random Forest Summary:')
for k, v in rf_summary.items():
    print(f'   {k:<30}: {v}')

  Training: Random Forest (streaming)
[Chunk   0] n=   15 | metric=0.0000 | cumulative=0.0000 | time=0.0266s | mem=0.6KB
[Chunk   1] n=   15 | metric=0.8667 | cumulative=0.4333 | time=0.0215s | mem=1.2KB
[Chunk   2] n=   15 | metric=0.7333 | cumulative=0.5333 | time=0.0311s | mem=1.8KB
[Chunk   3] n=   15 | metric=1.0000 | cumulative=0.6500 | time=0.0375s | mem=2.3KB
[Chunk   4] n=   15 | metric=0.8667 | cumulative=0.6933 | time=0.0470s | mem=2.9KB
[Chunk   5] n=   15 | metric=0.9333 | cumulative=0.7333 | time=0.0424s | mem=3.5KB
[Chunk   6] n=   15 | metric=0.8667 | cumulative=0.7524 | time=0.0588s | mem=4.1KB
[Chunk   7] n=   15 | metric=1.0000 | cumulative=0.7833 | time=0.0575s | mem=4.7KB
[Chunk   8] n=   15 | metric=0.8667 | cumulative=0.7926 | time=0.0630s | mem=5.3KB
[Chunk   9] n=   15 | metric=1.0000 | cumulative=0.8133 | time=0.0688s | mem=5.9KB

📈 Random Forest Summary:
   n_chunks                      : 10
   total_samples                 : 150
   mean_metric               

## 5. Visualise — Metric Over Time

Plot how accuracy improves as more chunks are seen.

In [6]:
matplotlib.use('Agg')

tree_acc = tree_trainer.get_metric_history('metric')
rf_acc   = rf_trainer.get_metric_history('metric')

# --- Plot 1: Decision Tree accuracy over chunks ---
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tree_acc, color='#2196F3', linewidth=2, marker='o',
        markersize=5, label='Decision Tree')
ax.fill_between(range(len(tree_acc)), tree_acc, alpha=0.15, color='#2196F3')
ax.set_title('Decision Tree — Accuracy Over Streaming Chunks',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Chunk Index')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle='--', alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig('tree_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: tree_accuracy.png')

✅ Saved: tree_accuracy.png


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/1586517500.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Visualise — Compare Decision Tree vs Random Forest

In [7]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(tree_acc, color='#2196F3', linewidth=2, marker='o',
        markersize=5, label='Decision Tree')
ax.fill_between(range(len(tree_acc)), tree_acc, alpha=0.12, color='#2196F3')

ax.plot(rf_acc, color='#E91E63', linewidth=2, marker='s',
        markersize=5, label='Random Forest (n=10)')
ax.fill_between(range(len(rf_acc)), rf_acc, alpha=0.12, color='#E91E63')

ax.set_title('Model Comparison — Streaming Accuracy Over Chunks',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Chunk Index')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: model_comparison.png')

# Print final comparison
print(f'\n🏆 Final Accuracy Comparison:')
print(f'   Decision Tree : {tree_summary["final_cumulative_metric"]:.4f}')
print(f'   Random Forest : {rf_summary["final_cumulative_metric"]:.4f}')
winner = 'Random Forest' if rf_summary['final_cumulative_metric'] >= tree_summary['final_cumulative_metric'] else 'Decision Tree'
print(f'   Winner        : {winner} 🏆')

✅ Saved: model_comparison.png

🏆 Final Accuracy Comparison:
   Decision Tree : 0.8600
   Random Forest : 0.8133
   Winner        : Decision Tree 🏆


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/3331411651.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Visualise — Predictions vs Ground Truth

In [8]:
# Predict on full dataset with both models
tree_preds = tree_model.predict(X_shuffled)
rf_preds   = rf_model.predict(X_shuffled)

# --- Tree predictions vs ground truth ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, preds, title, color in zip(
    axes,
    [tree_preds, rf_preds],
    ['Decision Tree', 'Random Forest'],
    ['#2196F3', '#E91E63']
):
    n = min(100, len(y_shuffled))
    correct = y_shuffled[:n] == preds[:n]
    idx_plot = np.arange(n)
    acc = correct.mean()

    ax.scatter(idx_plot[correct], y_shuffled[:n][correct],
               color='#4CAF50', s=30, label='Correct', alpha=0.8, zorder=3)
    ax.scatter(idx_plot[~correct], y_shuffled[:n][~correct],
               color='#F44336', s=40, marker='x', label='Wrong (true)', zorder=3)
    ax.scatter(idx_plot[~correct], preds[:n][~correct],
               color='#FF9800', s=40, marker='^', label='Wrong (pred)', zorder=3)

    ax.set_title(f'{title} | Accuracy={acc:.3f}', fontweight='bold')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Class Label')
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Setosa', 'Versicolor', 'Virginica'])
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Predictions vs Ground Truth', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('predictions_vs_truth.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: predictions_vs_truth.png')

✅ Saved: predictions_vs_truth.png


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/1467467584.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Streaming Metrics — Detailed Logging

In [9]:
# Track detailed streaming metrics
acc_metric  = StreamingAccuracy()
prf_metric  = StreamingPrecisionRecallF1(average='macro')
cm_metric   = StreamingConfusionMatrix(classes=[0, 1, 2])
roll_metric = RollingAccuracy(window_size=30)

acc_history  = []
roll_history = []
f1_history   = []

# Replay chunks through metrics
print('📊 Chunk-by-chunk Streaming Metrics (Random Forest):')
print(f'{"Chunk":>6} {"Acc":>8} {"RollAcc":>10} {"F1":>8}')
print('-' * 38)

# Re-train fresh model just for metrics demo
metric_model = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)

for i, (X_c, y_c) in enumerate(chunks):
    if i > 0:  # Need at least 1 chunk trained before predicting
        y_pred = metric_model.predict(X_c)
        acc_metric.update(y_c, y_pred)
        prf_metric.update(y_c, y_pred)
        cm_metric.update(y_c, y_pred)
        roll_metric.update(y_c, y_pred)

        acc_history.append(acc_metric.result())
        roll_history.append(roll_metric.result())
        f1_history.append(prf_metric.result()['f1'])

        print(f'{i:>6} {acc_metric.result():>8.4f} '
              f'{roll_metric.result():>10.4f} '
              f'{prf_metric.result()["f1"]:>8.4f}')

    metric_model.partial_fit(X_c, y_c)

print(f'\n✅ Final cumulative accuracy : {acc_metric.result():.4f}')
print(f'✅ Final rolling accuracy    : {roll_metric.result():.4f}')
print(f'✅ Final macro F1 score      : {prf_metric.result()["f1"]:.4f}')
prf_result = prf_metric.result()
print(f'✅ Precision                 : {prf_result["precision"]:.4f}')
print(f'✅ Recall                    : {prf_result["recall"]:.4f}')

📊 Chunk-by-chunk Streaming Metrics (Random Forest):
 Chunk      Acc    RollAcc       F1
--------------------------------------
     1   0.8667     0.8667   0.8500
     2   0.8000     0.8000   0.8000
     3   0.8667     0.8667   0.8697
     4   0.8667     0.9333   0.8704
     5   0.8800     0.9000   0.8792
     6   0.8778     0.9000   0.8817
     7   0.8952     0.9333   0.8972
     8   0.8917     0.9333   0.8908
     9   0.9037     0.9333   0.9026

✅ Final cumulative accuracy : 0.9037
✅ Final rolling accuracy    : 0.9333
✅ Final macro F1 score      : 0.9026
✅ Precision                 : 0.9029
✅ Recall                    : 0.9036


In [10]:
# Plot all metrics together
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics_data = [
    (acc_history,  'Cumulative Accuracy', '#2196F3'),
    (roll_history, 'Rolling Accuracy (w=30)', '#4CAF50'),
    (f1_history,   'Macro F1 Score', '#E91E63'),
]

for ax, (data, title, color) in zip(axes, metrics_data):
    x = np.arange(len(data))
    ax.plot(x, data, color=color, linewidth=2, marker='o', markersize=4)
    ax.fill_between(x, data, alpha=0.15, color=color)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Chunk')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Streaming Metrics Over Time — Random Forest on Iris',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('streaming_metrics.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: streaming_metrics.png')

✅ Saved: streaming_metrics.png


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/1309244121.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Confusion Matrix Visualisation

In [11]:
cm_array = cm_metric.result()
class_names = ['Setosa', 'Versicolor', 'Virginica']
n = len(class_names)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_array, cmap='Blues')
fig.colorbar(im, ax=ax)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(class_names, rotation=15)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Cumulative Confusion Matrix\n(Random Forest — Streaming)',
             fontsize=12, fontweight='bold')

thresh = cm_array.max() / 2.0
for i in range(n):
    for j in range(n):
        ax.text(j, i, str(cm_array[i, j]),
                ha='center', va='center', fontsize=13,
                color='white' if cm_array[i, j] > thresh else 'black')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrix.png')

✅ Saved: confusion_matrix.png


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/1069921880.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Pipeline Demo — Scaler + Model in One Chain

In [13]:
class MinMaxScaler:
    """Simple streaming MinMax scaler (no external libraries)."""
    def __init__(self):
        self._min = None
        self._max = None

    def partial_fit(self, X):
        mn = np.nanmin(X, axis=0)
        mx = np.nanmax(X, axis=0)
        if self._min is None:
            self._min = mn
            self._max = mx
        else:
            self._min = np.minimum(self._min, mn)
            self._max = np.maximum(self._max, mx)
        return self

    def transform(self, X):
        rng = self._max - self._min
        rng[rng == 0] = 1.0
        return (X - self._min) / rng


# Build streaming pipeline
pipe = StreamingPipeline([
    ('scaler', MinMaxScaler()),
    ('model',  RandomForestClassifier(n_estimators=10,
                                       max_depth=5,
                                       random_state=42))
])

print('🔧 Pipeline:', pipe)
print()

pipe_acc_history = []
pipe_acc = StreamingAccuracy()

for i, (X_c, y_c) in enumerate(chunks):
    if i > 0:
        y_pred = pipe.predict(X_c)
        pipe_acc.update(y_c, y_pred)
        pipe_acc_history.append(pipe_acc.result())
        print(f'  Chunk {i:2d}: accuracy = {pipe_acc.result():.4f}')
    pipe.partial_fit(X_c, y_c)

print(f'\n✅ Pipeline final accuracy: {pipe_acc.result():.4f}')

🔧 Pipeline: StreamingPipeline([scaler(MinMaxScaler) -> model(RandomForestClassifier)])

  Chunk  1: accuracy = 0.8667
  Chunk  2: accuracy = 0.8667
  Chunk  3: accuracy = 0.9111
  Chunk  4: accuracy = 0.9167
  Chunk  5: accuracy = 0.9200
  Chunk  6: accuracy = 0.9111
  Chunk  7: accuracy = 0.9048
  Chunk  8: accuracy = 0.9083
  Chunk  9: accuracy = 0.9185

✅ Pipeline final accuracy: 0.9185


## 11. Feature Importances

In [14]:
importances = rf_model.feature_importances_()

fig, ax = plt.subplots(figsize=(7, 4))
order = np.argsort(importances)[::-1]
sorted_imp = importances[order]
sorted_names = [feature_names[i] for i in order]

bars = ax.bar(range(len(sorted_imp)), sorted_imp,
              color=['#FF5722','#FF9800','#FFC107','#FFEB3B'])
ax.set_xticks(range(len(sorted_imp)))
ax.set_xticklabels(sorted_names, fontsize=11)
ax.set_title('Feature Importances — Random Forest on Iris',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Importance (normalised)')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)

for bar, val in zip(bars, sorted_imp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('feature_importances.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: feature_importances.png')

✅ Saved: feature_importances.png


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/2635519222.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Benchmark — Single Tree vs Random Forest

In [15]:
import time

def run_streaming_benchmark(model, chunks, model_name):
    times = []
    trainer = StreamTrainer(model, verbose=False)
    for X_c, y_c in chunks:
        t0 = time.perf_counter()
        trainer.fit_chunk(X_c, y_c)
        times.append(time.perf_counter() - t0)
    final_acc = trainer.summary().get('final_cumulative_metric', 0.0)
    mean_time = np.mean(times) * 1000
    total_time = np.sum(times)
    print(f'  {model_name:<35} | acc={final_acc:.4f} | '
          f'avg={mean_time:.2f}ms | total={total_time:.3f}s')
    return final_acc, mean_time

print('=' * 70)
print('  BENCHMARK: Streaming Model Comparison on Iris')
print('=' * 70)

results = {}
configs = [
    ('Decision Tree (depth=3)',
     DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Decision Tree (depth=5)',
     DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('Bagging (n=5, depth=3)',
     BaggingClassifier(n_estimators=5, max_depth=3, random_state=42)),
    ('Random Forest (n=5, depth=3)',
     RandomForestClassifier(n_estimators=5, max_depth=3, random_state=42)),
    ('Random Forest (n=10, depth=5)',
     RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)),
]

for name, model in configs:
    acc, ms = run_streaming_benchmark(model, chunks, name)
    results[name] = (acc, ms)

print('=' * 70)

  BENCHMARK: Streaming Model Comparison on Iris
  Decision Tree (depth=3)             | acc=0.8600 | avg=9.79ms | total=0.098s
  Decision Tree (depth=5)             | acc=0.8600 | avg=9.20ms | total=0.092s
  Bagging (n=5, depth=3)              | acc=0.8267 | avg=34.16ms | total=0.342s
  Random Forest (n=5, depth=3)        | acc=0.8000 | avg=18.85ms | total=0.188s
  Random Forest (n=10, depth=5)       | acc=0.8133 | avg=51.97ms | total=0.520s


In [16]:
# Benchmark bar chart
names = list(results.keys())
accs  = [v[0] for v in results.values()]
times = [v[1] for v in results.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3','#03A9F4','#FF9800','#E91E63','#9C27B0']

# Accuracy
bars1 = axes[0].bar(range(len(names)), accs, color=colors, alpha=0.85)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=20, ha='right', fontsize=8)
axes[0].set_title('Final Cumulative Accuracy', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, axis='y', linestyle='--', alpha=0.4)
for bar, val in zip(bars1, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=9)

# Time
bars2 = axes[1].bar(range(len(names)), times, color=colors, alpha=0.85)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, rotation=20, ha='right', fontsize=8)
axes[1].set_title('Average Time per Chunk (ms)', fontweight='bold')
axes[1].set_ylabel('Time (ms)')
axes[1].grid(True, axis='y', linestyle='--', alpha=0.4)
for bar, val in zip(bars2, times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}ms', ha='center', fontsize=9)

plt.suptitle('Streaming Benchmark — Accuracy vs Speed',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Saved: benchmark.png')

✅ Saved: benchmark.png


/var/folders/2m/9hdsdkjx5yvgz22g5_4xd6zw0000gn/T/ipykernel_25725/182011908.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Final Summary

In [17]:
print('=' * 60)
print('  NumCompute-Stream — Final Demo Summary')
print('=' * 60)
print(f'  Dataset          : Iris ({len(X)} samples, {X.shape[1]} features)')
print(f'  Streaming chunks : {len(chunks)} chunks of {CHUNK_SIZE} samples each')
print()
print(f'  Decision Tree    : {tree_summary["final_cumulative_metric"]:.4f} accuracy')
print(f'  Random Forest    : {rf_summary["final_cumulative_metric"]:.4f} accuracy')
print(f'  Pipeline         : {pipe_acc.result():.4f} accuracy')
print()
print('  Plots generated:')
plots = [
    'tree_accuracy.png',
    'model_comparison.png',
    'predictions_vs_truth.png',
    'streaming_metrics.png',
    'confusion_matrix.png',
    'feature_importances.png',
    'benchmark.png',
]
for p in plots:
    print(f'    ✅ {p}')
print()
print('  Framework: NumCompute-Stream')
print('  Libraries: NumPy + matplotlib ONLY')
print('=' * 60)

  NumCompute-Stream — Final Demo Summary
  Dataset          : Iris (150 samples, 4 features)
  Streaming chunks : 10 chunks of 15 samples each

  Decision Tree    : 0.8600 accuracy
  Random Forest    : 0.8133 accuracy
  Pipeline         : 0.9185 accuracy

  Plots generated:
    ✅ tree_accuracy.png
    ✅ model_comparison.png
    ✅ predictions_vs_truth.png
    ✅ streaming_metrics.png
    ✅ confusion_matrix.png
    ✅ feature_importances.png
    ✅ benchmark.png

  Framework: NumCompute-Stream
  Libraries: NumPy + matplotlib ONLY
